# 14 - Closed-loop test of the hard-negative detector (Kaggle)

Flies the same seeds as notebooks 11 and 12 with the detector chosen in notebook 13, judged under the same ten evidence rules, so the tables are directly comparable with the v2 results in `docs/validation.md` 6e/6f:
- **cost side:** seeds 0-299 (random; no hazard contact in them): needless back-aways per rule. v2 had 25 (L0) down to 4.
- **benefit side:** the 91 hazard seeds of notebook 12: hazard contacts per rule. v2 (L0) avoided 2 of 5 (complete map) and 6 of 9 (stale map).

Settings: **GPU T4 x2**, **Internet on**, add the notebook output of notebook 13 (Add Input, Notebook Output, search `notebook` + its id). Save Version, Save & Run All (Commit). Two flight processes run side by side; about 3 to 4 hours; they stop themselves at 5 h.

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /tmp/touch_down || (cd /tmp/touch_down && git pull -q)
!sh /tmp/touch_down/scripts/kaggle_setup.sh 2>&1 | tail -5

In [ ]:
import glob, os
chosen = glob.glob('/kaggle/input/**/chosen.txt', recursive=True)
assert chosen, 'add the output of notebook 13 as an input'
NAME = open(chosen[0]).read().strip()
CKPT = os.path.join(os.path.dirname(chosen[0]), NAME + '_best.pt')
assert os.path.getsize(CKPT) > 1e7, CKPT
print('detector:', NAME, CKPT)
HAZ_SEEDS = [4, 15, 19, 21, 24, 37, 64, 71, 79, 80, 88, 108, 111, 122, 125, 137, 146, 157, 162, 167, 168, 181, 184, 191, 194, 207, 234, 258, 270, 272, 291, 303, 324, 329, 354, 369, 372, 376, 383, 405, 420, 434, 435, 446, 449, 460, 477, 492, 494, 511, 514, 515, 525, 531, 532, 548, 559, 567, 576, 579, 588, 592, 593, 640, 664, 670, 680, 712, 728, 738, 741, 743, 772, 784, 801, 806, 818, 826, 838, 867, 873, 889, 894, 901, 913, 916, 917, 943, 957, 965, 996]
W = '/kaggle/working/hn_closed'
run = f'python /tmp/touch_down/scripts/kaggle_run.py --ckpt {CKPT} --nn-ablation --max-hours 5'
p1 = f"{run} --seed-list {' '.join(map(str, HAZ_SEEDS))} --out {W}/hazard && {run} --seeds 200 300 --out {W}/cost"
p2 = f"{run} --seeds 0 200 --out {W}/cost"

In [ ]:
import subprocess
procs = [subprocess.Popen(c, shell=True) for c in (p1, p2)]
for p in procs:
    p.wait()
print('both flight processes finished', [p.returncode for p in procs])

In [ ]:
# the two processes shared the cost folder, so recompute its tables from all landings
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python -m touchdown.analysis.ablation {W}/cost --out {W}/cost/ablation.json
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python -m touchdown.analysis.compare_published {W}/cost --out {W}/cost/summary.json | tail -3
!ls {W}/cost | wc -l; ls {W}/hazard | wc -l